# Capítulo 6 — Percolação em Meio Poroso

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Aplicar** a Lei de Darcy para fluxo em meio saturado e não-saturado
2. **Calcular** carga hidráulica e gradientes hidráulicos em perfis de solo
3. **Estimar** parâmetros de van Genuchten ($\theta_r$, $\theta_s$, $\alpha$, $n$, $K_{sat}$) a partir da textura do solo
4. **Resolver** a equação de Richards 1D para infiltração vertical em regime permanente
5. **Interpretar** a influência da anisotropia de condutividade hidráulica no fluxo lateral
6. **Implementar** esquema implícito por Diferenças Finitas para transporte de umidade

---

## 📚 Conteúdo Teórico Resumido

### 6.1 Lei de Darcy e Carga Hidráulica

O fluxo de água em meio poroso saturado é governado pela **Lei de Darcy**:

$$
q = -K \cdot \nabla h \quad \text{[m/s]}
$$

onde:
- $q$ = fluxo de Darcy (velocidade aparente) [m/s]
- $K$ = condutividade hidráulica saturada [m/s]
- $h$ = carga hidráulica total [m]

A **carga hidráulica** $h$ é a soma da carga de posição ($z$) e da carga de pressão ($\psi$):

$$
h = z + \psi \quad \text{[m]}
$$

### 6.2 Modelo de van Genuchten-Mualem

A relação entre conteúdo de água $\theta$, potencial matricial $\psi$ e condutividade hidráulica $K$ em meio não-saturado é descrita pelo modelo empírico de **van Genuchten-Mualem**:

$$
\theta(\psi) = \theta_r + \frac{\theta_s - \theta_r}{\left[1 + (\alpha \cdot |\psi|)^n\right]^m}, \quad m = 1 - \frac{1}{n}
$$

$$
K(\theta) = K_{sat} \cdot S_e^L \cdot \left[1 - \left(1 - S_e^{1/m}\right)^m\right]^2, \quad S_e = \frac{\theta - \theta_r}{\theta_s - \theta_r}
$$

### 6.3 Equação de Richards

Combinando a Lei de Darcy generalizada com a equação da continuidade, obtém-se a **equação de Richards** para fluxo vertical 1D:

$$
\frac{\partial \theta}{\partial t} = \frac{\partial}{\partial z} \left[ K(\theta) \cdot \left( \frac{\partial \psi}{\partial z} + 1 \right) \right] \quad \text{[s$^{-1}$]}
$$

### 6.4 Esquema Implícito por Diferenças Finitas

Para fins didáticos, consideramos uma **linearização local** em torno de um valor de referência $\theta_0$:

$$
\frac{\partial \theta}{\partial t} + u_0 \frac{\partial \theta}{\partial x} = D_0 \frac{\partial^2 \theta}{\partial x^2}
$$

O esquema implícito avalia as derivadas espaciais no instante $n+1$:

$$
-\alpha_D \theta_{i-1}^{n+1} + (1 + 2\alpha_D + \alpha_A) \theta_i^{n+1} - (\alpha_D - \alpha_A) \theta_{i+1}^{n+1} = \theta_i^n
$$

onde $\alpha_A = u_0 \Delta t / \Delta x$ e $\alpha_D = D_0 \Delta t / \Delta x^2$.

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os

# Adiciona o diretório raiz ao path para importar módulos
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.linalg import solve_banded

# Configuração de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | SciPy {__import__('scipy').__version__}")

---

## 🔬 Seção 1: Lei de Darcy e Carga Hidráulica

### Exercício 1.1: Carga Hidráulica e Sentido do Fluxo

Calcule a carga hidráulica $h$ em um ponto do solo onde a cota é $z = -2,0$ m e o potencial matricial é $\psi = -0,5$ m. Qual é o sentido do fluxo se em outro ponto ($z = -3,0$ m, $\psi = 0$ m) a carga é maior?

In [ ]:
# ============================================================
# EXERCÍCIO 1.1: Carga Hidráulica
# ============================================================

print("=" * 60)
print("EXERCÍCIO 1.1: CARGA HIDRÁULICA E SENTIDO DO FLUXO")
print("=" * 60)

# Dados do problema
z1 = -2.0    # m (cota do ponto 1)
psi1 = -0.5  # m (potencial matricial do ponto 1)

z2 = -3.0    # m (cota do ponto 2)
psi2 = 0.0   # m (potencial matricial do ponto 2 - saturado)

# Cálculo da carga hidráulica
h1 = z1 + psi1
h2 = z2 + psi2

print(f"\n📊 Dados do problema:")
print(f"  • Ponto 1: z₁ = {z1} m, ψ₁ = {psi1} m")
print(f"  • Ponto 2: z₂ = {z2} m, ψ₂ = {psi2} m")

print(f"\n🧮 Cálculo da carga hidráulica:")
print(f"  • h₁ = z₁ + ψ₁ = {z1} + ({psi1}) = {h1} m")
print(f"  • h₂ = z₂ + ψ₂ = {z2} + ({psi2}) = {h2} m")

print(f"\n🔄 Sentido do fluxo:")
if h1 > h2:
    print(f"  • h₁ ({h1} m) > h₂ ({h2} m)")
    print(f"  • Fluxo do ponto 1 para o ponto 2 (DESCENDENTE)")
else:
    print(f"  • h₁ ({h1} m) < h₂ ({h2} m)")
    print(f"  • Fluxo do ponto 2 para o ponto 1 (ASCENDENTE)")

print(f"\n💡 Interpretação:")
print(f"  A água flui de regiões de MAIOR carga hidráulica para")
print(f"  regiões de MENOR carga hidráulica, independentemente")
print(f"  da cota ou do potencial matricial isoladamente.")

---

## 🌊 Seção 2: Modelo de van Genuchten-Mualem

### Exercício 2.1: Curva de Retenção

Implemente a função de van Genuchten e plote a curva de retenção $\theta(\psi)$ para um solo franco-arenoso.

In [ ]:
# ============================================================
# EXERCÍCIO 2.1: Curva de Retenção de van Genuchten
# ============================================================

def van_genuchten_theta(psi, theta_r, theta_s, alpha, n):
    """
    Calcula o conteúdo de água θ em função do potencial matricial ψ.
    
    Parâmetros:
    -----------
    psi : float ou array
        Potencial matricial [m]. Negativos em solo não-saturado.
    theta_r : float
        Conteúdo de água residual [m³/m³]
    theta_s : float
        Conteúdo de água na saturação [m³/m³]
    alpha : float
        Parâmetro inverso da pressão de entrada de ar [m⁻¹]
    n : float
        Parâmetro de distribuição de tamanho de poros
    
    Retorna:
    --------
    float ou array
        Conteúdo de água θ [m³/m³]
    """
    m = 1 - 1/n
    psi = np.atleast_1d(psi)
    theta = np.zeros_like(psi)
    
    # Para ψ >= 0 (saturado)
    theta[psi >= 0] = theta_s
    
    # Para ψ < 0 (não-saturado)
    mask = psi < 0
    psi_neg = np.abs(psi[mask])
    theta[mask] = theta_r + (theta_s - theta_r) / (1 + (alpha * psi_neg)**n)**m
    
    return theta

def van_genuchten_K(theta, theta_r, theta_s, K_sat, n, L=0.5):
    """
    Calcula a condutividade hidráulica K em função do conteúdo de água θ.
    
    Parâmetros:
    -----------
    theta : float ou array
        Conteúdo de água [m³/m³]
    theta_r : float
        Conteúdo de água residual [m³/m³]
    theta_s : float
        Conteúdo de água na saturação [m³/m³]
    K_sat : float
        Condutividade hidráulica saturada [m/s]
    n : float
        Parâmetro de distribuição de tamanho de poros
    L : float, opcional
        Parâmetro de conectividade de poros (padrão: 0,5)
    
    Retorna:
    --------
    float ou array
        Condutividade hidráulica K [m/s]
    """
    m = 1 - 1/n
    theta = np.atleast_1d(theta)
    Se = (theta - theta_r) / (theta_s - theta_r)
    Se = np.clip(Se, 0, 1)  # Limitar entre 0 e 1
    
    K = K_sat * Se**L * (1 - (1 - Se**(1/m))**m)**2
    
    return K

# Parâmetros do solo franco-arenoso (Rosetta)
theta_r = 0.058    # m³/m³
theta_s = 0.41     # m³/m³
alpha = 7.4        # m⁻¹
n = 1.56
K_sat = 5.1e-5     # m/s (4,4 m/dia)

print("=" * 60)
print("CURVA DE RETENÇÃO DE VAN GENUCHTEN")
print("=" * 60)

print(f"\n📊 Parâmetros do solo franco-arenoso:")
print(f"  • θ_r = {theta_r} m³/m³")
print(f"  • θ_s = {theta_s} m³/m³")
print(f"  • α = {alpha} m⁻¹")
print(f"  • n = {n}")
print(f"  • m = 1 - 1/n = {1 - 1/n:.3f}")
print(f"  • K_sat = {K_sat:.2e} m/s ({K_sat*86400:.1f} m/dia)")

# Gerar curva de retenção
psi_range = np.logspace(-4, 0, 100)  # de 0,0001 a 1 m (valores absolutos)
psi_range = -psi_range  # Tornar negativo (sucção)
theta_range = van_genuchten_theta(psi_range, theta_r, theta_s, alpha, n)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.abs(psi_range), theta_range, 'b-', linewidth=2)
ax.set_xlabel('|ψ| [m]', fontsize=12)
ax.set_ylabel('θ [m³/m³]', fontsize=12)
ax.set_title('Curva de Retenção - Solo Franco-Arenoso', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.axhline(y=theta_s, color='r', linestyle='--', label=f'θ_s = {theta_s}')
ax.axhline(y=theta_r, color='g', linestyle='--', label=f'θ_r = {theta_r}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\n💡 Interpretação:")
print(f"  • Para |ψ| → 0 (solo saturado): θ → θ_s = {theta_s}")
print(f"  • Para |ψ| → ∞ (solo seco): θ → θ_r = {theta_r}")
print(f"  • A curva tem forma sigmoidal (S invertido)")

---

## 📊 Seção 3: Condutividade Hidráulica Não-Saturada

### Exercício 3.1: Curva K(θ)

Plote a condutividade hidráulica $K(\theta)$ em função do conteúdo de água.

In [ ]:
# ============================================================
# EXERCÍCIO 3.1: Condutividade Hidráulica K(θ)
# ============================================================

print("=" * 60)
print("CONDUTIVIDADE HIDRÁULICA NÃO-SATURADA")
print("=" * 60)

# Gerar curva K(θ)
theta_range_K = np.linspace(theta_r + 0.001, theta_s, 100)
K_range = van_genuchten_K(theta_range_K, theta_r, theta_s, K_sat, n)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(theta_range_K, K_range, 'r-', linewidth=2)
ax.set_xlabel('θ [m³/m³]', fontsize=12)
ax.set_ylabel('K [m/s]', fontsize=12)
ax.set_title('Condutividade Hidráulica - Solo Franco-Arenoso', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.axvline(x=theta_s, color='b', linestyle='--', label=f'θ_s = {theta_s}')
ax.axhline(y=K_sat, color='g', linestyle='--', label=f'K_sat = {K_sat:.2e} m/s')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Calcular K para θ = 0,25 m³/m³
theta_teste = 0.25
K_teste = van_genuchten_K(theta_teste, theta_r, theta_s, K_sat, n)

print(f"\n📊 Condutividade para θ = {theta_teste} m³/m³:")
print(f"  • K = {K_teste:.2e} m/s")
print(f"  • K/K_sat = {K_teste/K_sat:.3f} ({K_teste/K_sat*100:.1f}%)")

print(f"\n💡 Interpretação:")
print(f"  • A condutividade K varia exponencialmente com θ")
print(f"  • Para θ próximo de θ_s: K ≈ K_sat")
print(f"  • Para θ próximo de θ_r: K → 0 (várias ordens de grandeza)")
print(f"  • Essa não-linearidade exige solução numérica da equação de Richards")

---

## 🧮 Seção 4: Esquema Implícito por Diferenças Finitas

### Exercício 4.1: Montagem da Matriz Tridiagonal

Para um solo com $u_0 = 0,05$ cm/h, $D_0 = 0,2$ cm²/h, discretize com $\Delta x = 1$ cm e $\Delta t = 2$ h. Calcule os coeficientes $a_i$, $b_i$, $c_i$ da matriz tridiagonal.

In [ ]:
# ============================================================
# EXERCÍCIO 4.1: Montagem da Matriz Tridiagonal
# ============================================================

print("=" * 60)
print("ESQUEMA IMPLÍCITO - MATRIZ TRIDIAGONAL")
print("=" * 60)

# Dados do problema
u0 = 0.05       # cm/h (velocidade de Darcy)
D0 = 0.2        # cm²/h (difusividade hidráulica)
dx = 1.0        # cm (espaçamento espacial)
dt = 2.0        # h (passo de tempo)
N = 20          # número de células

# Coeficientes adimensionais
alpha_A = u0 * dt / dx
alpha_D = D0 * dt / dx**2

print(f"\n📊 Dados do problema:")
print(f"  • u₀ = {u0} cm/h")
print(f"  • D₀ = {D0} cm²/h")
print(f"  • Δx = {dx} cm")
print(f"  • Δt = {dt} h")
print(f"  • N = {N} células")

print(f"\n🧮 Coeficientes adimensionais:")
print(f"  • α_A = u₀·Δt/Δx = {u0}×{dt}/{dx} = {alpha_A:.3f}")
print(f"  • α_D = D₀·Δt/Δx² = {D0}×{dt}/{dx}² = {alpha_D:.3f}")

# Coeficientes da matriz
a = -alpha_D * np.ones(N)           # subdiagonal
b = (1 + 2*alpha_D + alpha_A) * np.ones(N)  # diagonal principal
c = -(alpha_D - alpha_A) * np.ones(N)       # superdiagonal

print(f"\n📋 Coeficientes da matriz tridiagonal:")
print(f"  • a_i (subdiagonal) = {a[0]:.3f}")
print(f"  • b_i (diagonal principal) = {b[0]:.3f}")
print(f"  • c_i (superdiagonal) = {c[0]:.3f}")

# Verificar dominância diagonal
dominancia = abs(b[0]) - abs(a[0]) - abs(c[0])

print(f"\n✅ Verificação de dominância diagonal:")
print(f"  • |b_i| - |a_i| - |c_i| = {abs(b[0]):.3f} - {abs(a[0]):.3f} - {abs(c[0]):.3f} = {dominancia:.3f}")
if dominancia > 0:
    print(f"  • Matriz DIAGONALMENTE DOMINANTE → esquema estável ✓")
else:
    print(f"  • Matriz NÃO diagonalmente dominante → verificar estabilidade")

print(f"\n💡 Interpretação:")
print(f"  • O esquema implícito é incondicionalmente estável")
print(f"  • A dominância diagonal garante convergência do algoritmo de Thomas")
print(f"  • Para problemas não-lineares, atualize D(θ) e K(θ) a cada passo")

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia ambiental.

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 6.1** | Percolação em Terreno com Textura Conhecida | [🔗 Abrir](../casos/06_1_Percolacao_Terreno_Textura_Conhecida.ipynb) |
| **Caso 6.2** | Curva de Retenção de van Genuchten | [🔗 Abrir](../casos/06_2_Curva_Retencao_van_Genuchten.ipynb) |

> 💡 **Dica:** Os estudos de caso podem ser executados independentemente deste notebook principal.

---

## 📖 Referências

- van GENUCHTEN, M. Th. (1980). A closed-form equation for predicting the hydraulic conductivity of unsaturated soils. *Soil Science Society of America Journal*, 44(5), 892–898.
- MUALEM, Y. (1976). A new model for predicting the hydraulic conductivity of unsaturated porous media. *Water Resources Research*, 12(3), 513–522.
- SCHAP, M. G.; LEIJ, F. J.; van GENUCHTEN, M. Th. (2001). Rosetta: a computer program for estimating soil hydraulic properties with hierarchical pedotransfer functions. *Journal of Hydrology*, 251(3-4), 163–176.
- BEAR, J. (1972). *Dynamics of Fluids in Porous Media*. Elsevier.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Hidrodinâmica de Canais](05_Hidrodinamica_Canais_Abertos.ipynb)
- [📚 Próximo Capítulo: Fundamentos de Transferência de Calor](07_Fundamentos_Calor.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 6**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print("=" * 60)
print("✅ NOTEBOOK CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Capítulo 6!")
print("📖 Próximo passo: Capítulo 7 - Fundamentos de Transferência de Calor")
print("\n💡 Dica: Execute este notebook quantas vezes precisar!")
print("   Modifique os parâmetros e explore diferentes cenários.")